# COMPASS Engine: Interactive Demo

## Clinical Ontology-driven Multi-modal Predictive Agentic Support System

This notebook shows how to run COMPASS on a participant folder and inspect outputs.

What you'll do:
- Optionally launch the web dashboard
- Run the pipeline in-notebook (CLI mode)
- Inspect the final reports (standard + deep phenotyping)

Prerequisites:
- Data under `src/full_stack/backend/data/pseudo_data` (demo)
- If using OpenRouter, set `OPENROUTER_API_KEY` (recommended)
- Optional fallback: set `OPENAI_API_KEY` to allow automatic public API fallback


In [1]:
import sys
import json
import ipywidgets as widgets
from IPython.display import display, Markdown, JSON, clear_output
from pathlib import Path
import subprocess
import time


def find_repo_root(start: Path) -> Path:
    for path in [start] + list(start.parents):
        backend = path / "src" / "full_stack" / "backend"
        if (path / "main.py").exists() and (backend / "agents").exists():
            return path
    return start


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"[*] Repo Root: {repo_root}")

from main import run_compass_pipeline
from src.full_stack.backend.config.settings import get_settings, LLMBackend
from src.full_stack.backend.data.models.prediction_task import build_task_spec_from_flat_args, PredictionTaskSpec

PSEUDO_DATA_ROOT = repo_root / "src" / "full_stack" / "backend" / "data" / "pseudo_data" / "inputs"

print(f"[OK] COMPASS Engine Loaded. Data Source: {PSEUDO_DATA_ROOT}")


[*] Repo Root: ~/compass_pipeline/multi_agent_decision_support_system
[OK] COMPASS Engine Loaded. Data Source: ~/compass_pipeline/multi_agent_decision_support_system/src/full_stack/backend/data/pseudo_data


## Quick Launch: Web Dashboard

Launch the live UI (`main.py --ui`) for step-by-step monitoring in your browser.
This uses the same Python environment as the notebook.


In [2]:
def launch_dashboard(b):
    clear_output()
    print("Launching dashboard on port 5005...")
    print("Open: http://127.0.0.1:5005")
    try:
        main_script = repo_root / "main.py"
        subprocess.Popen([sys.executable, str(main_script), "--ui"])
    except Exception as e:
        print(f"Error launching: {e}")

btn = widgets.Button(description="Launch Web Dashboard", button_style='info', icon='play')
btn.on_click(launch_dashboard)
display(btn)


Button(button_style='info', description='Launch Web Dashboard', icon='play', style=ButtonStyle())

## Interactive Notebook Execution

Run the pipeline directly in the notebook (CLI mode).
This is useful for quick iteration and debugging without the web dashboard.


### 1. Configuration
Select a subject, prediction type, mode-specific outputs, backend, and number of iterations.
Public API mode uses OpenRouter model routing; Local mode uses your local model runtime on this machine.


In [ ]:
# Find available subjects
subjects = [d.name for d in PSEUDO_DATA_ROOT.iterdir() if d.is_dir() and not d.name.startswith('.')]
subjects.sort()

# Interactive Widgets
subject_dropdown = widgets.Dropdown(
    options=subjects,
    description='Subject:',
)

prediction_type_dropdown = widgets.Dropdown(
    options=[
        ('Binary classification', 'binary'),
        ('Multi-class classification', 'multiclass'),
        ('Univariate regression', 'regression_univariate'),
        ('Multivariate regression', 'regression_multivariate'),
        ('Hierarchical (JSON task spec)', 'hierarchical'),
    ],
    value='binary',
    description='Type:',
)

target_input = widgets.Text(
    value='target_phenotype',
    description='Target:',
    placeholder='Task label or output profile label'
)

control_input = widgets.Text(
    value='non-target comparator phenotype profile',
    description='Control:',
    placeholder='Binary comparator label',
)

class_labels_input = widgets.Text(
    value='subtype_a,subtype_b,subtype_c',
    description='Classes:',
    placeholder='multiclass labels (comma-separated)',
    layout=widgets.Layout(display='none'),
)

regression_outputs_input = widgets.Text(
    value='total_iq',
    description='Outputs:',
    placeholder='regression outputs (comma-separated)',
    layout=widgets.Layout(display='none'),
)

task_spec_json_input = widgets.Textarea(
    value='',
    description='Task JSON:',
    placeholder='Required when Type=hierarchical. Paste canonical task spec JSON.',
    layout=widgets.Layout(display='none', width='100%', height='180px'),
)

iterations_input = widgets.BoundedIntText(
    value=2,
    min=1,
    max=5,
    description='Iterations:',
)

# Backend Selection
backend_dropdown = widgets.Dropdown(
    options=[('Public API (OpenRouter)', 'openrouter'), ('Local LLM (vLLM/Transformers)', 'local')],
    value='openrouter',
    description='Backend:',
)

model_input = widgets.Text(
    value='Qwen/Qwen2.5-0.5B-Instruct',
    description='Local Model:',
    disabled=True
)


def on_backend_change(change):
    model_input.disabled = change['new'] != 'local'


def on_prediction_type_change(change):
    mode = change['new']
    control_input.layout.display = '' if mode == 'binary' else 'none'
    class_labels_input.layout.display = '' if mode == 'multiclass' else 'none'
    regression_outputs_input.layout.display = '' if mode in {'regression_univariate', 'regression_multivariate'} else 'none'
    task_spec_json_input.layout.display = '' if mode == 'hierarchical' else 'none'


backend_dropdown.observe(on_backend_change, names='value')
prediction_type_dropdown.observe(on_prediction_type_change, names='value')
on_prediction_type_change({'new': prediction_type_dropdown.value})

display(
    subject_dropdown,
    prediction_type_dropdown,
    target_input,
    control_input,
    class_labels_input,
    regression_outputs_input,
    task_spec_json_input,
    iterations_input,
    backend_dropdown,
    model_input,
)


### 2. Run Analysis
This executes the pipeline and prints verbose output so you can follow agent reasoning.


In [ ]:
PARTICIPANT_DIR = PSEUDO_DATA_ROOT / subject_dropdown.value
PREDICTION_TYPE = prediction_type_dropdown.value
TARGET = (target_input.value or '').strip() or 'target_phenotype'
CONTROL = (control_input.value or '').strip()
CLASS_LABELS = [x.strip() for x in (class_labels_input.value or '').split(',') if x.strip()]
REGRESSION_OUTPUTS = [x.strip() for x in (regression_outputs_input.value or '').split(',') if x.strip()]
TASK_SPEC_JSON = (task_spec_json_input.value or '').strip()
MAX_ITERS = iterations_input.value

print(f"Selected Participant: {PARTICIPANT_DIR}")

# Apply Backend Settings
settings = get_settings()
if backend_dropdown.value == 'local':
    settings.models.backend = LLMBackend.LOCAL
    settings.models.local_model_name = model_input.value
    print(f"Configured for LOCAL inference using {model_input.value}")
else:
    settings.models.backend = LLMBackend.OPENROUTER
    for attr in [
        'orchestrator_model', 'critic_model', 'predictor_model',
        'integrator_model', 'communicator_model', 'tool_model'
    ]:
        setattr(settings.models, attr, 'google/gemini-3.1-flash-lite')
    print("Configured for OpenRouter inference (google/gemini-3.1-flash-lite for all agents)")

if PREDICTION_TYPE == 'hierarchical':
    if not TASK_SPEC_JSON:
        raise ValueError('Hierarchical mode requires task_spec_json_input.')
    prediction_task_spec = PredictionTaskSpec(**json.loads(TASK_SPEC_JSON))
else:
    prediction_task_spec = build_task_spec_from_flat_args(
        prediction_type=PREDICTION_TYPE,
        target_label=TARGET,
        control_label=CONTROL or 'CONTROL',
        class_labels=CLASS_LABELS,
        regression_outputs=REGRESSION_OUTPUTS,
    )

legacy_target, legacy_control = prediction_task_spec.legacy_target_control()
print(f"Starting analysis for: {legacy_target} ({prediction_task_spec.root.mode.value})")
if prediction_task_spec.root.mode.value == 'binary_classification':
    print(f"Control comparator: {legacy_control}")

# Run Pipeline
try:
    result = run_compass_pipeline(
        participant_dir=PARTICIPANT_DIR,
        target_condition=legacy_target,
        control_condition=legacy_control,
        prediction_task_spec=prediction_task_spec,
        max_iterations=MAX_ITERS,
        verbose=True,
        interactive_ui=False  # Use CLI output mode for notebook compatibility
    )
except Exception as e:
    print(f"Execution Error: {e}")
    result = None


### 3. Inspector
Review the final standard report, the deep phenotyping report, and the structured execution log generated by the run.


In [ ]:
if result:
    log_path = Path(result['output_dir']) / f"execution_log_{result['participant_id']}.json"
    report_path = Path(result['output_dir']) / f"report_{result['participant_id']}.md"
    deep_path = Path(result['output_dir']) / "deep_phenotype.md"

    if isinstance(result.get('probability'), (int, float)):
        print(f"RESULT: {result['prediction']} ({result['probability']:.1%})")
    else:
        print(f"RESULT: {result['prediction']}")

    if report_path.exists():
        display(Markdown("### Final Clinical Report"))
        with open(report_path, 'r') as f:
            display(Markdown(f.read()))

    if deep_path.exists():
        display(Markdown("### Deep Phenotyping Report"))
        with open(deep_path, 'r') as f:
            display(Markdown(f.read()))

    if log_path.exists():
        display(Markdown("### Execution Log"))
        with open(log_path, 'r') as f:
            display(JSON(json.load(f), expanded=False))
else:
    print("No successful result to inspect.")


## Explainability (XAI) CLI options

The three backend explainability methods are:
- `external` (aHFR-TokenSHAP): hierarchy-constrained Shapley sampling over feature leaves, using binary decision score (`CASE` vs `CONTROL`) and adaptive subtree focus.
- `internal` (IGA): integrated gradients on the predictor-style prompt; token attributions are aggregated to feature value spans and L1-normalized across features.
- `hybrid` (LLM-select): feature-name prior importance (no participant values) using phenotype + feature names only.

Execution policy: methods run once on the **selected final attempt** and write `xai_feature_importance_<participant_id>.json`.

Optional: add `--generate_xai_report` to run the Communicator over XAI outputs and generate `xai_explainability_report.md`.


In [ ]:
# Example command with all XAI methods enabled
participant = "src/full_stack/backend/data/pseudo_data/inputs/SUBJ_001_PSEUDO"
cmd = [
    sys.executable,
    str(repo_root / "main.py"),
    participant,
    "--prediction_type", "binary",
    "--target_label", "Major Depressive Disorder",
    "--control_label", "Comparator",
    "--backend", "openrouter",
    "--xai_methods", "external,internal,hybrid",
    "--xai_external_k", "4",
    "--xai_internal_model", "Qwen/Qwen2.5-0.5B-Instruct",
    "--xai_internal_steps", "8",
    "--xai_hybrid_model", "google/gemini-3.1-flash-lite",
    "--generate_xai_report",
]
print(" ".join(cmd))
# subprocess.run(cmd, check=True)
